# <font color="#418FDE" size="6.5" uppercase>**Neighbor Prediction**</font>

>Last update: 20260713.
    
By the end of this Lecture, you will be able to:
- Use nearest-neighbor search with brute-force, KD-tree, and Ball-tree strategies. 
- Fit K-neighbor and radius-neighbor classifiers and regressors. 
- Analyze how distance metrics, weighting, and scaling affect neighbor predictions. 


## **1. Neighbor Search Methods**

### **1.1. Unsupervised Neighbor Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_01_01.jpg?v=1784000602" width="250">



>* Find closest data points using distance metrics
>* Retrieve similar examples without predicting labels

>* Fit structures to store and organize data
>* Query fixed neighbors or radius-based matches

>* Scale features and choose suitable distance metrics
>* Use trees to speed large searches



In [ ]:
#@title Python Code - Unsupervised Neighbor Search

# This example compares unsupervised neighbor search strategies.
# NearestNeighbors stores points and answers similarity queries.
# The output shows matching neighbors across algorithms.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import sklearn

# Small two-dimensional points keep distances easy to visualize.
points = np.array([
    [1.0, 1.0], [1.5, 1.2], [2.0, 1.8], [5.0, 5.0],
    [5.5, 5.2], [6.0, 5.8], [8.0, 1.0], [8.5, 1.4]
])

query_point = np.array([[5.2, 5.1]])
neighbor_count = 3

# Validate the simple shape expected by the neighbor search.
if points.shape[1] != query_point.shape[1]:
    raise ValueError("Points and query must have the same feature count.")

# Each algorithm searches the same stored points without labels.
algorithms = ["brute", "kd_tree", "ball_tree"]
results = {}

for algorithm in algorithms:
    searcher = NearestNeighbors(n_neighbors=neighbor_count, algorithm=algorithm)
    searcher.fit(points)
    distances, indices = searcher.kneighbors(query_point)
    results[algorithm] = (distances[0], indices[0])

print("scikit-learn version:", sklearn.__version__)
print("Query point:", query_point[0].round(2).tolist())

for algorithm in algorithms:
    distances, indices = results[algorithm]
    print(algorithm, "indices:", indices.tolist(), "distances:", distances.round(2).tolist())

# Plot the brute-force result because all strategies agree here.
brute_distances, brute_indices = results["brute"]
neighbor_points = points[brute_indices]

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(points[:, 0], points[:, 1], label="stored points")
ax.scatter(query_point[:, 0], query_point[:, 1], marker="X", s=120, label="query")
ax.scatter(neighbor_points[:, 0], neighbor_points[:, 1], s=120, facecolors="none", edgecolors="red", label="nearest neighbors")
ax.set_title("Unsupervised nearest-neighbor search")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.show()



### **1.2. Brute Force Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_01_02.jpg?v=1784000606" width="250">



>* Compares each query with every stored point
>* Simple, exact, and assumption-free neighbor search

>* Large datasets require many distance calculations
>* High dimensions can favor brute force

>* Exact, flexible search for many metrics
>* Useful for small, changing, high-dimensional data



In [ ]:
#@title Python Code - Brute Force Search

# This example demonstrates brute force neighbor search.
# Every query compares against all stored points.
# The output identifies exact nearest neighbors.

import numpy as np
from sklearn.neighbors import NearestNeighbors
import sklearn

# Small two-feature data keeps every distance easy to inspect.
stored_points = np.array([[1.0, 1.0], [2.0, 1.5], [3.0, 3.5], [6.0, 5.0]])
query_point = np.array([[2.5, 2.0]])

# Validate the simple shape expected by nearest-neighbor search.
if stored_points.shape[1] != query_point.shape[1]:
    raise ValueError("Stored points and query point need matching features.")

# Brute force checks every stored point using Euclidean distance.
brute_model = NearestNeighbors(n_neighbors=2, algorithm="brute", metric="euclidean")
brute_model.fit(stored_points)

# The model returns distances and row positions of nearest neighbors.
distances, neighbor_indices = brute_model.kneighbors(query_point)
nearest_rows = neighbor_indices[0]

print("scikit-learn version:", sklearn.__version__)
print("Query point:", query_point[0].tolist())
print("Nearest row indices:", nearest_rows.tolist())
print("Nearest distances:", np.round(distances[0], 3).tolist())
print("Nearest points:", stored_points[nearest_rows].tolist())



### **1.3. Tree Based Search**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_01_03.jpg?v=1784000604" width="250">



>* Organizes data before nearest-neighbor queries
>* Skips irrelevant regions to search faster

>* KD-trees split data along feature axes
>* High dimensions reduce KD-tree search efficiency

>* Ball-trees group data into searchable nested spheres
>* Choose methods by data, metrics, and query costs



In [ ]:
#@title Python Code - Tree Based Search

# Compare tree based nearest neighbor search methods.
# KD trees and Ball trees organize points differently.
# The output shows identical neighbors and query costs.

import numpy as np
import sklearn
from sklearn.datasets import make_blobs
from sklearn.neighbors import NearestNeighbors

# Create a small two dimensional dataset for visualization-free clarity.
points, labels = make_blobs(
    n_samples=300,
    centers=4,
    n_features=2,
    random_state=42,
)

# Validate the simple shape expected by this lesson.
if points.shape != (300, 2):
    raise ValueError("Expected 300 rows and 2 numeric features.")

# Use one query point near the middle of the cloud.
query_point = np.array([[0.0, 0.0]])
neighbor_count = 5

# Fit the same neighbor search with three algorithms.
algorithms = ["brute", "kd_tree", "ball_tree"]
results = []

for algorithm in algorithms:
    searcher = NearestNeighbors(
        n_neighbors=neighbor_count,
        algorithm=algorithm,
    )
    searcher.fit(points)
    distances, indices = searcher.kneighbors(query_point)
    results.append((algorithm, distances[0], indices[0]))

# Print concise results that compare the search strategies.
print("scikit-learn version:", sklearn.__version__)
print("Query point:", query_point[0].round(2).tolist())

for algorithm, distances, indices in results:
    rounded_distances = distances.round(2).tolist()
    neighbor_ids = indices.tolist()
    print(algorithm, "distances", rounded_distances, "indices", neighbor_ids)

# Confirm that tree search returns the same nearest points here.
kd_matches = np.array_equal(results[0][2], results[1][2])
ball_matches = np.array_equal(results[0][2], results[2][2])
print("Tree methods match brute-force neighbors:", kd_matches and ball_matches)



## **2. Neighbor Models**

### **2.1. KNN Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_02_01.jpg?v=1784000608" width="250">



>* Classifies new cases using nearby labeled examples
>* K controls sensitivity versus prediction stability

>* Stores training examples for later comparison
>* Predicts using neighbor votes or weights

>* Scale features and choose distances carefully
>* Tune K to balance noise and detail



In [ ]:
#@title Python Code - KNN Classification

# This example fits a KNN classifier.
# Scaling keeps distance comparisons fair.
# Accuracy and one prediction show results.

import numpy as np
import sklearn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small built-in classification dataset.
iris = load_iris()
features = iris.data
target = iris.target

# Validate the basic shape before modeling.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows must match target labels.")

# Split data while preserving class proportions.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Build a scaled KNN classifier with five neighbors.
knn_model = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=5)
)

# Fit stores the training examples for neighbor lookup.
knn_model.fit(X_train, y_train)

# Evaluate predictions on unseen test flowers.
accuracy = knn_model.score(X_test, y_test)
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])
predicted_class = knn_model.predict(new_flower)[0]

print(f"scikit-learn version: {sklearn.__version__}")
print(f"Test accuracy with scaled 5-NN: {accuracy:.2f}")
print(f"New flower prediction: {iris.target_names[predicted_class]}")



### **2.2. Radius Classification**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_02_02.jpg?v=1784000610" width="250">



>* Classifies using examples within a chosen radius
>* Neighbor count varies by local data density

>* Store data and choose a search radius
>* Nearby labels vote on each prediction

>* Choose radius carefully and scale features
>* Data density can guide useful predictions



In [ ]:
#@title Python Code - Radius Classification

# This example fits a radius-neighbor classifier.
# The radius controls which neighbors can vote.
# The output compares small and larger radii.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.neighbors import RadiusNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Create a small two-class dataset with clear clusters.
features, labels = make_blobs(
    n_samples=160,
    centers=2,
    cluster_std=1.25,
    random_state=42,
)

# Split data so accuracy is measured on unseen points.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    labels,
    test_size=0.25,
    stratify=labels,
    random_state=42,
)

# Validate that the split kept matching feature and label counts.
if X_train.shape[0] != y_train.shape[0]:
    raise ValueError("Training features and labels must match.")

# Fit one radius classifier with a deliberately small radius.
small_radius_model = make_pipeline(
    StandardScaler(),
    RadiusNeighborsClassifier(radius=0.35, outlier_label=0),
)
small_radius_model.fit(X_train, y_train)

# Fit another radius classifier that can include more neighbors.
large_radius_model = make_pipeline(
    StandardScaler(),
    RadiusNeighborsClassifier(radius=0.85, outlier_label=0),
)
large_radius_model.fit(X_train, y_train)

# Count neighbors around one test point after scaling.
scaled_train = small_radius_model.named_steps["standardscaler"].transform(X_train)
scaled_test = small_radius_model.named_steps["standardscaler"].transform(X_test)
example_point = scaled_test[[0]]

small_neighbors = np.linalg.norm(scaled_train - example_point, axis=1) <= 0.35
large_neighbors = np.linalg.norm(scaled_train - example_point, axis=1) <= 0.85

# Print concise results that show the radius effect.
print("scikit-learn version:", sklearn.__version__)
print("Small radius accuracy:", round(small_radius_model.score(X_test, y_test), 3))
print("Large radius accuracy:", round(large_radius_model.score(X_test, y_test), 3))
print("Neighbors for one test point:", int(small_neighbors.sum()), "vs", int(large_neighbors.sum()))

# Plot the training data and the two radius circles.
fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(scaled_train[:, 0], scaled_train[:, 1], c=y_train, cmap="coolwarm")
ax.scatter(example_point[:, 0], example_point[:, 1], c="black", marker="x", s=90)

small_circle = plt.Circle(example_point[0], 0.35, fill=False, color="black")
large_circle = plt.Circle(example_point[0], 0.85, fill=False, color="green")
ax.add_patch(small_circle)
ax.add_patch(large_circle)

ax.set_title("Radius classification neighborhoods")
ax.set_xlabel("Scaled feature 1")
ax.set_ylabel("Scaled feature 2")
ax.legend(handles=scatter.legend_elements()[0], labels=["Class 0", "Class 1"])
plt.show()



### **2.3. KNN Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_02_03.jpg?v=1784000612" width="250">



>* Predict numbers from nearby similar examples
>* Fit by storing data for neighbor search

>* Few neighbors capture detail but amplify noise
>* More neighbors smooth predictions but may overgeneralize

>* Weight closer neighbors more heavily
>* Scale features and choose distances carefully



In [ ]:
#@title Python Code - KNN Regression

# Demonstrate KNN regression on simple housing-like data.
# Show how scaling protects distance-based neighbor choices.
# Compare unscaled and scaled prediction accuracy clearly.

import numpy as np
import sklearn
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

# Create a small deterministic regression dataset.
features, prices = make_regression(
    n_samples=220,
    n_features=2,
    noise=18,
    random_state=42,
)

# Put one feature on a much larger scale.
features[:, 0] = features[:, 0] * 1000 + 2000
features[:, 1] = features[:, 1] * 2 + 6
prices = prices + 300

# Check that the generated arrays match correctly.
if features.shape != (220, 2):
    raise ValueError("Expected 220 rows and 2 feature columns.")

# Split data before fitting any preprocessing step.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    prices,
    test_size=0.25,
    random_state=42,
)

# Fit KNN directly on the original feature scales.
unscaled_model = KNeighborsRegressor(n_neighbors=5)
unscaled_model.fit(X_train, y_train)
unscaled_predictions = unscaled_model.predict(X_test)

# Fit KNN after standardizing features inside a pipeline.
scaled_model = make_pipeline(
    StandardScaler(),
    KNeighborsRegressor(n_neighbors=5),
)
scaled_model.fit(X_train, y_train)
scaled_predictions = scaled_model.predict(X_test)

# Evaluate both models with mean absolute error.
unscaled_mae = mean_absolute_error(y_test, unscaled_predictions)
scaled_mae = mean_absolute_error(y_test, scaled_predictions)

# Predict one new home-like example with both approaches.
new_home = np.array([[2300, 7.5]])
unscaled_price = unscaled_model.predict(new_home)[0]
scaled_price = scaled_model.predict(new_home)[0]

print("scikit-learn version:", sklearn.__version__)
print("Unscaled KNN MAE:", round(unscaled_mae, 2))
print("Scaled KNN MAE:", round(scaled_mae, 2))
print("Unscaled prediction for new case:", round(unscaled_price, 2))
print("Scaled prediction for new case:", round(scaled_price, 2))



## **3. Metrics and Weights**

### **3.1. Radius Based Regression**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_03_01.jpg?v=1784000614" width="250">



>* Radius defines which examples guide prediction
>* Small versus large radius trades precision for stability

>* Scale features so radius reflects true similarity
>* Distance metrics change which neighbors count

>* Weight closer neighbors more heavily
>* Handle sparse areas with careful radius choices



In [ ]:
#@title Python Code - Radius Based Regression

# Demonstrate radius based regression with scaled features.
# Compare equal and distance weighted neighbor predictions.
# Show how radius changes local averaging behavior.

import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.neighbors import RadiusNeighborsRegressor
from sklearn.preprocessing import StandardScaler

# Small synthetic housing data keeps the lesson focused.
size_sq_m = np.array([45, 50, 55, 60, 70, 80, 90, 100, 115, 130], dtype=float)
age_years = np.array([35, 30, 28, 25, 20, 18, 15, 12, 10, 8], dtype=float)
rent_euros = np.array([900, 980, 1050, 1120, 1280, 1450, 1600, 1780, 2050, 2350])

# Scaling makes size and age comparable for distance.
features = np.column_stack((size_sq_m, age_years))
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# The query apartment is transformed using the same scaler.
query_home = np.array([[85.0, 16.0]])
scaled_query = scaler.transform(query_home)

# Two radii show narrow versus broad local evidence.
small_radius = 0.75
large_radius = 1.60

# Equal weights average all neighbors inside the radius.
equal_small = RadiusNeighborsRegressor(radius=small_radius, weights="uniform")
equal_large = RadiusNeighborsRegressor(radius=large_radius, weights="uniform")
equal_small.fit(scaled_features, rent_euros)
equal_large.fit(scaled_features, rent_euros)

# Distance weights favor the closest neighbors inside the radius.
weighted_large = RadiusNeighborsRegressor(radius=large_radius, weights="distance")
weighted_large.fit(scaled_features, rent_euros)

# Count neighbors to connect radius choice with prediction stability.
small_neighbors = equal_small.radius_neighbors(scaled_query, return_distance=False)[0]
large_neighbors = equal_large.radius_neighbors(scaled_query, return_distance=False)[0]

if len(small_neighbors) == 0 or len(large_neighbors) == 0:
    raise ValueError("Each radius should find at least one neighbor.")

small_prediction = equal_small.predict(scaled_query)[0]
large_prediction = equal_large.predict(scaled_query)[0]
weighted_prediction = weighted_large.predict(scaled_query)[0]

print("scikit-learn version:", sklearn.__version__)
print("Query apartment: 85 sq m and 16 years old")
print("Small radius neighbors:", len(small_neighbors))
print("Large radius neighbors:", len(large_neighbors))
print("Small radius equal-weight rent: €" + str(round(small_prediction)))
print("Large radius equal-weight rent: €" + str(round(large_prediction)))
print("Large radius distance-weight rent: €" + str(round(weighted_prediction)))

# Plot size against rent and highlight included neighbors.
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(size_sq_m, rent_euros, color="lightgray", label="training homes")
ax.scatter(size_sq_m[large_neighbors], rent_euros[large_neighbors], color="tab:blue", label="large radius")
ax.scatter(size_sq_m[small_neighbors], rent_euros[small_neighbors], color="tab:orange", label="small radius")
ax.scatter(query_home[0, 0], weighted_prediction, color="black", marker="X", s=90, label="query prediction")

ax.set_title("Radius based regression uses all points within a distance")
ax.set_xlabel("Apartment size (square meters)")
ax.set_ylabel("Monthly rent (euros)")
ax.legend()
plt.show()



### **3.2. Distance Weighted Predictions**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_03_02.jpg?v=1784000616" width="250">



>* Closer neighbors influence predictions more
>* Applies to both classes and values

>* Closer neighbors guide mixed-region predictions
>* Weights preserve local patterns in regression

>* Scale features so distances stay meaningful
>* Check noise, outliers, metrics, and neighbors



In [ ]:
#@title Python Code - Distance Weighted Predictions

# This example compares neighbor voting weights.
# Closer neighbors can influence predictions more.
# The plot shows a changed decision boundary.

import numpy as np
import matplotlib.pyplot as plt
from sklearn import __version__ as sklearn_version
from sklearn.neighbors import KNeighborsClassifier

# Small two-feature data keeps distances easy to inspect.
X = np.array([[0.0, 0.0], [0.2, 0.1], [0.4, -0.1], [1.0, 0.0], [1.2, 0.1]])
y = np.array([0, 0, 0, 1, 1])

# The query is closest to class one.
query = np.array([[0.9, 0.0]])
if X.shape != (5, 2) or y.shape[0] != X.shape[0]:
    raise ValueError("Expected five two-feature training examples.")

# Both models use the same neighbors but different weights.
uniform_model = KNeighborsClassifier(n_neighbors=5, weights="uniform")
distance_model = KNeighborsClassifier(n_neighbors=5, weights="distance")

uniform_model.fit(X, y)
distance_model.fit(X, y)

# Distances explain why the weighted prediction can differ.
distances = np.linalg.norm(X - query, axis=1)
uniform_prediction = int(uniform_model.predict(query)[0])
distance_prediction = int(distance_model.predict(query)[0])

print(f"scikit-learn version: {sklearn_version}")
print(f"Uniform prediction: class {uniform_prediction}")
print(f"Distance-weighted prediction: class {distance_prediction}")
print(f"Nearest distances: {np.round(np.sort(distances)[:5], 2).tolist()}")

# A one-axis plot shows the same query and training points.
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X[y == 0, 0], X[y == 0, 1], label="class 0", s=80)
ax.scatter(X[y == 1, 0], X[y == 1, 1], label="class 1", s=80)
ax.scatter(query[:, 0], query[:, 1], marker="*", s=220, label="query")

ax.set_title("Distance weighting favors closer neighbors")
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend()
plt.show()



### **3.3. Choosing Distance Metrics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_13/Lecture_A/image_03_03.jpg?v=1784000618" width="250">



>* Distance metrics define meaningful neighbor similarity
>* Poor metrics create misleading comparisons

>* Match metrics to data relationships
>* Use direction or overlap for sparse patterns

>* Scale and weight features before measuring distance
>* Validate metrics against predictions and domain knowledge



In [ ]:
#@title Python Code - Choosing Distance Metrics

# Compare distance metrics for neighbor prediction.
# Scaling changes which neighbors look closest.
# Validation accuracy reveals the better metric.

import numpy as np
import sklearn
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Load a small numeric classification dataset.
wine = load_wine()
features = wine.data
target = wine.target

# Check the dataset has matching rows and labels.
if features.shape[0] != target.shape[0]:
    raise ValueError("Feature rows and labels must match.")

# Split data before scaling to avoid leakage.
X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, stratify=target, random_state=42
)

# Try two common metrics with and without scaling.
experiments = [
    ("Euclidean, unscaled", "euclidean", False),
    ("Manhattan, unscaled", "manhattan", False),
    ("Euclidean, scaled", "euclidean", True),
    ("Manhattan, scaled", "manhattan", True),
]

print("scikit-learn version:", sklearn.__version__)
print("KNN accuracy on the wine dataset:")

# Fit each simple KNN pipeline and compare accuracy.
for label, metric, use_scaling in experiments:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    model = make_pipeline(StandardScaler(), knn) if use_scaling else knn
    model.fit(X_train, y_train)
    accuracy = model.score(X_test, y_test)
    print(label + ":", round(accuracy, 3))

# Inspect one test point to connect metrics with neighbors.
scaled_model = make_pipeline(
    StandardScaler(), KNeighborsClassifier(n_neighbors=5, metric="manhattan")
)
scaled_model.fit(X_train, y_train)
prediction = scaled_model.predict(X_test[:1])[0]
print("First test prediction with scaled Manhattan:", wine.target_names[prediction])



# <font color="#418FDE" size="6.5" uppercase>**Neighbor Prediction**</font>


In this lecture, you learned to:
- Use nearest-neighbor search with brute-force, KD-tree, and Ball-tree strategies. 
- Fit K-neighbor and radius-neighbor classifiers and regressors. 
- Analyze how distance metrics, weighting, and scaling affect neighbor predictions. 

In the next Lecture (Lecture B), we will go over 'Advanced Neighbors'